# Learning curves

- `modelling_df4_dimless_p3-NN_2.3-opt_MLP_simple_search.ipynb`: две ячейки после "Save same models, but on CPU" [по одной на каждый таргет]
- `modelling_df4_dimless_p3-NN_3.2-opt_TabNet_splashing_narrow_search.ipynb`: поледняя ячейка
- `modelling_df4_dimless_p3-NN_3.3-opt_TabNet_no_fragmentation_narrow_search.ipynb`: последняя ячейка

# Import libraries

In [1]:
# Changes to all modules will automatically be applied when any cell runs. 
%load_ext autoreload
%autoreload 2

In [2]:
import logging

# Set logging level for PyTorch Tabular
logging.getLogger("pytorch_tabular").setLevel(logging.ERROR)

# Set logging level for PyTorch Lightning
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)

# Set logging level for Lightning Fabric
logging.getLogger("lightning_fabric").setLevel(logging.ERROR)

# Disable device availability messages
logging.getLogger("lightning.pytorch.utilities.rank_zero").setLevel(logging.FATAL)

import pandas as pd
import numpy as np

from pytorch_tabular.models import CategoryEmbeddingModelConfig, TabNetModelConfig

import optuna

from functools import partial

from pathlib import Path
import sys

from typing import Type
from sklearn.base import BaseEstimator

from sklearn.metrics import (
    make_scorer,
    f1_score,
)

sys.path.append(
    str(Path('..', 'utils_functionality', 'split_utils'))
)
sys.path.append(
    str(Path('..', 'utils_functionality', 'models'))
)

from split_tools import get_train_test
from modelling4_utils import (
    # _prepare_df,
    MLPipeline,
    StatsModelsEstimator,
    PytorchTabularEstimator,
    deep_update_estimator_params,
    OptunaOptimizer,
    GridSearchOptimizer,
    RANDOM_STATE,
)

# from pytorch_lightning.loggers import MLFlowLogger
from pytorch_lightning.loggers import TensorBoardLogger

seed = RANDOM_STATE

# Fix error with save weights

# import torch
# # from omegaconf import DictConfig
# # from omegaconf.base import ContainerMetadata
# # import typing
# # torch.serialization.add_safe_globals([DictConfig, ContainerMetadata, typing.Any])
# import pytorch_tabular.utils.python_utils as pt_utils
# pt_utils.pl_load = lambda f, map_location: torch.load(f, map_location=map_location, weights_only=False)



In [3]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore")

# Settings

In [4]:
save_model_and_metrics = False
model_postfix = 'opt-model'
metrics_file = ''

## MLP

### Splashing

In [5]:
estimator = PytorchTabularEstimator
target = "splashing"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'dropout': 0.04379983378244369,
        'learning_rate': 0.009487772841911948,
        'layers': '128',
        'activation': 'LeakyReLU',
        'batch_norm_continuous_input': True, # We already have normalized features    
        'use_batch_norm': False, # For hidden layers
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 300, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[CV] END .................................................... total time=   0.3s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.2s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.3s
[CV] END .................................................... total time=   0.3s
no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,CategoryEmbeddingModel_splashing_smote_opt-model
holdout_test_f1_macro,0.836601
holdout_test_accuracy_balanced,0.828704
holdout_test_roc_auc,0.911265
holdout_test_f1,0.888889
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.860788
cv_test_accuracy_balanced_median,0.873839
cv_test_roc_auc_median,0.919505


### No fragmentation

In [6]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": CategoryEmbeddingModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'dropout': 0.4124773666300698,
        'learning_rate': 0.012011447873654357,
        'layers': '128-64',
        'activation': 'LeakyReLU',
        'batch_norm_continuous_input': True, # We already have normalized features    
        'use_batch_norm': False, # For hidden layers
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 300, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.4s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.6s
[CV] END .................................................... total time=   0.5s
[CV] END .................................................... total time=   0.4s
no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,CategoryEmbeddingModel_no_fragmentation_smoten...
holdout_test_f1_macro,0.834656
holdout_test_accuracy_balanced,0.845455
holdout_test_roc_auc,0.95
holdout_test_f1,0.761905
holdout_test_accuracy,0.866667
cv_test_f1_macro_median,0.898077
cv_test_accuracy_balanced_median,0.915751
cv_test_roc_auc_median,0.981685


## TabNet

### Splashing

In [7]:
estimator = PytorchTabularEstimator
target = "splashing"
model_postfix = 'opt-model'
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

add_smote = True
is_smotenc = False
smote_params = {
    # 'categorical_features': (
    #     'wettability',
    #     'inclination',
    # ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'n_d': 12,
        'n_a': 4,
        'n_steps': 2,
        'gamma': 1.4781750460004004,
        'n_shared': 1,
        'n_independent': 2,
        'learning_rate': 0.003182186027654311,
        'virtual_batch_size': 16,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 64, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "splashing" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_splashing.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(),
                                 ('inclination', 'volume_fraction')),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability',))])

[CV] END .................................................... total time=   2.3s
[CV] END .................................................... total time=   3.6s
[CV] END .................................................... total time=   1.1s
[CV] END .................................................... total time=   2.2s
[CV] END .................................................... total time=   1.5s
[CV] END .................................................... total time=   2.6s
[CV] END .................................................... total time=   2.7s
no summary in estimator class "PytorchTabularEstimator"


,0
target,splashing
model,TabNetModel_splashing_smote_opt-model
holdout_test_f1_macro,0.839525
holdout_test_accuracy_balanced,0.836806
holdout_test_roc_auc,0.936728
holdout_test_f1,0.886598
holdout_test_accuracy,0.853333
cv_test_f1_macro_median,0.805718
cv_test_accuracy_balanced_median,0.829721
cv_test_roc_auc_median,0.893189


### No fragmentation

In [8]:
estimator = PytorchTabularEstimator
target = "no_fragmentation"
model_postfix = 'opt-model'
metrics_file = "metrics_modelling4_6-neural_networks.xlsx"

add_smote = True
is_smotenc = True
smote_params = {
    'categorical_features': (
        'wettability',
        'inclination',
    ),
}

estimator_params = {
    "model_class": TabNetModelConfig,
    # Other params SET in optuna
    "model_config_params": {
        'n_d': 4,
        'n_a': 8,
        'n_steps': 2,
        'gamma': 1.722004270893931,
        'n_shared': 1,
        'n_independent': 2,
        'learning_rate': 0.00781783794374976,
        'virtual_batch_size': 16,
    },
    # "data_config_params": {},
    "trainer_config_params": {
        'max_epochs': 200,
        'batch_size': 64, # let's condiser whole dataset
        'early_stopping': 'valid_loss',
        'early_stopping_patience': 10,
        'accelerator': 'cpu',
        'progress_bar': 'none',
    },
    "optimizer_config_params": {
        'optimizer': 'Adam',
        'lr_scheduler': 'ReduceLROnPlateau',
        'lr_scheduler_params': {
            'patience': 5,
            'factor': 0.5,
        },
    },
}

ml_pipe = MLPipeline(
    target=target,
    estimator=estimator,
    estimator_params=estimator_params,
    model_postfix=model_postfix,
    add_smote=add_smote,
    is_smotenc=is_smotenc,
    smote_params=smote_params,
    metrics_file=metrics_file,
)

ml_pipe.run(
    save_model_and_metrics=save_model_and_metrics,
)

Load dataset from: ../data/df_dimless.xlsx
Keep "no_fragmentation" from {'splashing', 'no_fragmentation'}
Load split indexes from: ../data/df_ml_split_no_fragmentation.xlsx
std_features


('sedimentation_Stk',
 'particle_droplet_diameter_ratio',
 'relative_roughness',
 'K')

ColumnTransformer(transformers=[('minmax', MinMaxScaler(), ()),
                                ('std', StandardScaler(),
                                 ('sedimentation_Stk',
                                  'particle_droplet_diameter_ratio',
                                  'relative_roughness', 'K')),
                                ('passthrough', 'passthrough',
                                 ('wettability', 'inclination',
                                  'volume_fraction'))])

[CV] END .................................................... total time=   2.2s
[CV] END .................................................... total time=   3.8s
[CV] END .................................................... total time=   2.2s
[CV] END .................................................... total time=   2.9s
[CV] END .................................................... total time=   2.0s
[CV] END .................................................... total time=   2.5s
[CV] END .................................................... total time=   2.8s
no summary in estimator class "PytorchTabularEstimator"


,0
target,no_fragmentation
model,TabNetModel_no_fragmentation_smotenc_opt-model
holdout_test_f1_macro,0.863609
holdout_test_accuracy_balanced,0.918182
holdout_test_roc_auc,0.952727
holdout_test_f1,0.816327
holdout_test_accuracy,0.88
cv_test_f1_macro_median,0.871224
cv_test_accuracy_balanced_median,0.89011
cv_test_roc_auc_median,0.943223
